# Day 2 student exercise: improve a spam classifier

Use the SMS messages from the guided Day 2 notebook. The setup supplies a fixed train/test split, a dictionary baseline, and a TF–IDF logistic-regression baseline. **Edit the cells marked STUDENT EDIT** and compare each edited version with its own baseline.

Work in this order: change a dictionary using **training messages only**; change one TF–IDF setting; evaluate all four versions on the same held-out messages; read their errors; write a short interpretation. Do not choose more changes after looking at the held-out results unless you clearly describe that as exploration.

## 1. Prepared setup — run without editing

The **human label** is `spam` or `ham` (ordinary message). The positive class in all metrics is `spam`. `min_df=2` keeps a feature only if it appears in at least two training messages; `ngram_range=(1, 2)` includes single words and adjacent word pairs. The vectorizer is fitted on training messages only.

In [ ]:
# Import the tools used in the guided example and in this exercise.
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import spacy
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score,
                             precision_score, recall_score, f1_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline


def find_data_dir():
    # Run from materials/notebooks/ or from the repository root.
    for candidate in (Path('../data'), Path('materials/data'), Path('data')):
        if (candidate / 'sms_spam.csv').exists():
            return candidate
    raise FileNotFoundError('Could not find materials/data/sms_spam.csv')


DATA_DIR = find_data_dir()
try:
    nlp = spacy.load('en_core_web_sm')
except OSError:
    # The blank tokenizer is enough for the exercise; no model download is needed.
    nlp = spacy.blank('en')


def spacy_preprocess(text, remove_stop=True, min_len=2):
    # Match the guided notebook's token rules: lowercase alphabetic tokens.
    tokens = []
    for token in nlp.make_doc(str(text)):
        if not token.is_alpha or (remove_stop and token.is_stop):
            continue
        term = token.text.lower()
        if len(term) >= min_len:
            tokens.append(term)
    return tokens


sms = pd.read_csv(DATA_DIR / 'sms_spam.csv')[['text', 'label']].dropna().copy()
sms['label'] = sms['label'].str.lower()
train_df, test_df = train_test_split(
    sms, test_size=0.25, random_state=42, stratify=sms['label']
)
print('Training:', len(train_df), 'Held-out:', len(test_df))
display(train_df['label'].value_counts().rename('training messages'))

In [ ]:
# A fixed starting dictionary, as in the guided notebook.
spam_words = {
    'free', 'win', 'winner', 'won', 'prize', 'cash', 'urgent', 'claim',
    'txt', 'reply', 'call', 'mobile', 'ringtone', 'stop', 'guaranteed'
}


def dictionary_predict(text, words, threshold=1):
    # Each matching token adds one point; one point is enough to flag spam.
    tokens = spacy_preprocess(text, remove_stop=False, min_len=1)
    score = sum(token in words for token in tokens)
    return 'spam' if score >= threshold else 'ham'


baseline_dict_pred = test_df['text'].apply(
    lambda text: dictionary_predict(text, spam_words)
)


def make_model(min_df=2, ngram_range=(1, 2)):
    # tokenizer= leaves sklearn in control of ngram_range; analyzer= would not.
    return Pipeline([
        ('vectorizer', TfidfVectorizer(
            tokenizer=spacy_preprocess, token_pattern=None, lowercase=False,
            min_df=min_df, ngram_range=ngram_range, max_features=7000
        )),
        ('classifier', LogisticRegression(
            max_iter=1000, class_weight='balanced', random_state=42
        ))
    ])


baseline_model = make_model()
baseline_model.fit(train_df['text'], train_df['label'])
baseline_model_pred = baseline_model.predict(test_df['text'])
print('Both baselines are ready.')

## 2. STUDENT EDIT 1 — change the dictionary

Read the **training** examples below. Add at least two plausible spam terms to `student_words` and remove at least one questionable term. For example, you might inspect whether `call` or `stop` occurs in ordinary messages too. The cell must produce `student_dict_pred` for the held-out messages. Do not look at the held-out text while choosing words.

In [ ]:
# Inspect only training examples while choosing dictionary words.
display(train_df.groupby('label', group_keys=False).sample(n=5, random_state=42)[['label', 'text']])

In [ ]:
# STUDENT EDIT 1: add at least two terms and remove at least one.
student_words = spam_words.copy()
# TODO: student_words.update({'term_one', 'term_two'})
# TODO: student_words.discard('questionable_term')

assert len(student_words - spam_words) >= 2, 'Add at least two terms.'
assert len(spam_words - student_words) >= 1, 'Remove at least one term.'

# TODO: apply dictionary_predict with student_words to test_df['text'].
student_dict_pred = None
assert student_dict_pred is not None, 'Create student_dict_pred.'
print('Added:', sorted(student_words - spam_words))
print('Removed:', sorted(spam_words - student_words))

## 3. STUDENT EDIT 2 — change one TF–IDF setting

The baseline uses `min_df=2` and `ngram_range=(1, 2)`. Change **exactly one**: try `min_df=5` to drop rarer features, or `ngram_range=(1, 1)` to use only single words. Then complete the `.fit()` and `.predict()` lines. Both pipelines train on the same training messages.

In [ ]:
# STUDENT EDIT 2: change exactly one value below.
student_min_df = 2
student_ngram_range = (1, 2)

assert sum((student_min_df != 2, student_ngram_range != (1, 2))) == 1, (
    'Change min_df OR ngram_range, not both.'
)
student_model = make_model(
    min_df=student_min_df, ngram_range=student_ngram_range
)

# TODO: fit student_model on train_df['text'] and train_df['label'].
# TODO: predict test_df['text'] and assign the result to student_model_pred.
student_model_pred = None
assert student_model_pred is not None, 'Create student_model_pred.'
print('Baseline features:', len(baseline_model.named_steps['vectorizer'].get_feature_names_out()))
print('Edited features:', len(student_model.named_steps['vectorizer'].get_feature_names_out()))

## 4. STUDENT EDIT 3 — compare on the same held-out messages

Complete the two missing metric calls. `precision`, `recall`, and `f1` all treat **spam** as the positive class. The dictionary and the classifier answer the same task, but each pair isolates one decision you changed.

In [ ]:
def evaluate(y_true, y_pred):
    # Compare predictions to the same human labels in every row.
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, pos_label='spam', zero_division=0),
        'recall': recall_score(y_true, y_pred, pos_label='spam', zero_division=0),
        'f1': f1_score(y_true, y_pred, pos_label='spam', zero_division=0),
    }


baseline_dict_metrics = evaluate(test_df['label'], baseline_dict_pred)
baseline_model_metrics = evaluate(test_df['label'], baseline_model_pred)
# STUDENT EDIT 3: call evaluate for each edited prediction series.
student_dict_metrics = None  # TODO
student_model_metrics = None  # TODO
assert student_dict_metrics is not None and student_model_metrics is not None

comparison = pd.DataFrame.from_dict({
    'dictionary baseline': baseline_dict_metrics,
    'edited dictionary': student_dict_metrics,
    'TF-IDF logistic baseline': baseline_model_metrics,
    'edited TF-IDF logistic': student_model_metrics,
}, orient='index')
display(comparison.round(3))

## 5. Inspect errors

The next cells draw confusion matrices and show **up to five** false positives and false negatives for each edited version. There may be fewer than five of a given type. An ordinary message wrongly flagged as spam is a **false positive**; missed spam is a **false negative**.

In [ ]:
# Both matrices use the held-out labels and the same class order.
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
for ax, predictions, title in (
    (axes[0], student_dict_pred, 'Edited dictionary'),
    (axes[1], student_model_pred, 'Edited TF-IDF logistic'),
):
    ConfusionMatrixDisplay.from_predictions(
        test_df['label'], predictions, labels=['ham', 'spam'], ax=ax,
        colorbar=False
    )
    ax.set_title(title)
plt.tight_layout()


def error_examples(predictions, kind, n=5):
    # Keep the original message, human label, and model label together.
    rows = test_df[['text', 'label']].copy()
    rows['prediction'] = pd.Series(predictions, index=test_df.index)
    if kind == 'false_positive':
        rows = rows[(rows['label'] == 'ham') & (rows['prediction'] == 'spam')]
    elif kind == 'false_negative':
        rows = rows[(rows['label'] == 'spam') & (rows['prediction'] == 'ham')]
    else:
        raise ValueError('Choose false_positive or false_negative.')
    return rows.head(n)


for name, predictions in (
    ('Edited dictionary', student_dict_pred),
    ('Edited TF-IDF logistic', student_model_pred),
):
    print(name, '— false positives')
    display(error_examples(predictions, 'false_positive'))
    print(name, '— false negatives')
    display(error_examples(predictions, 'false_negative'))

## 6. Discuss what changed

Write four or five sentences below:

1. Which words did you add/remove, and which TF–IDF setting did you change?
2. Compared with each baseline, what happened to spam **precision** and **recall**? Give numbers.
3. Describe one actual false positive and one false negative. What caused them?
4. Which version would you trust more for this task, and what would you check before using its predictions in research?

**Your discussion:**